In [ ]:
# Generate structured reports
print("\n" + "="*80)
print("GENERATING REPORTS")
print("="*80 + "\n")

# Create report generator
report_gen = ReportGenerator(df, dataset_name="Sales Transaction Data")

# Generate markdown report
markdown_report = report_gen.generate_markdown_report('../results/EDA_Report.md')
print("✓ Markdown report generated: results/EDA_Report.md")

# Generate text report
text_report = report_gen.generate_text_report('../results/EDA_Report.txt')
print("✓ Text report generated: results/EDA_Report.txt")

print("\n" + "="*80)
print("EDA ANALYSIS COMPLETE")
print("="*80)
print("\nGenerated Files:")
print("  • Visualizations: 08 PNG files in results/")
print("  • Reports: Markdown and Text formats in results/")
print("\nAll analysis artifacts are saved in the results/ directory.")

In [ ]:
# Generate comprehensive insights report
print("\n" + "="*80)
print("EXPLORATORY DATA ANALYSIS - KEY FINDINGS & INSIGHTS")
print("="*80 + "\n")

# 1. Dataset Overview
print("1. DATASET OVERVIEW")
print("-" * 80)
print(f"• Total Records: {len(df):,}")
print(f"• Total Features: {len(df.columns)}")
print(f"• Date Range: {df['purchase_date'].min().date()} to {df['purchase_date'].max().date()}")
print(f"• Data Quality: No missing values detected ✓")

# 2. Customer Demographics
print("\n2. CUSTOMER DEMOGRAPHICS")
print("-" * 80)
print(f"• Age Range: {df['age'].min()} - {df['age'].max()} years")
print(f"• Average Age: {df['age'].mean():.1f} years")
print(f"• Gender Distribution: {dict(df['gender'].value_counts())}")
print(f"• Geographic Coverage: {df['location'].nunique()} unique locations")

# 3. Purchase Behavior
print("\n3. PURCHASE BEHAVIOR")
print("-" * 80)
print(f"• Average Purchase Amount: ${df['purchase_amount'].mean():.2f}")
print(f"• Median Purchase Amount: ${df['purchase_amount'].median():.2f}")
print(f"• Purchase Amount Range: ${df['purchase_amount'].min():.2f} - ${df['purchase_amount'].max():.2f}")
print(f"• Average Quantity: {df['quantity'].mean():.2f} units")
print(f"• Total Revenue: ${df['purchase_amount'].sum():.2f}")

# 4. Product Category Performance
print("\n4. PRODUCT CATEGORY PERFORMANCE")
print("-" * 80)
category_analysis = df.groupby('product_category').agg({
    'purchase_amount': ['sum', 'mean', 'count'],
    'satisfaction_score': 'mean'
}).round(2)
for category in df['product_category'].unique():
    cat_data = df[df['product_category'] == category]
    print(f"• {category}:")
    print(f"  - Count: {len(cat_data)} | Avg Amount: ${cat_data['purchase_amount'].mean():.2f} | " +
          f"Total: ${cat_data['purchase_amount'].sum():.2f} | Avg Satisfaction: {cat_data['satisfaction_score'].mean():.2f}")

# 5. Payment Method Analysis
print("\n5. PAYMENT METHOD ANALYSIS")
print("-" * 80)
payment_analysis = df.groupby('payment_method')['purchase_amount'].agg(['count', 'mean', 'sum']).round(2)
for method in df['payment_method'].unique():
    method_data = df[df['payment_method'] == method]
    print(f"• {method}: {len(method_data)} transactions | " +
          f"Avg: ${method_data['purchase_amount'].mean():.2f} | Total: ${method_data['purchase_amount'].sum():.2f}")

# 6. Customer Loyalty
print("\n6. CUSTOMER LOYALTY & SATISFACTION")
print("-" * 80)
print(f"• Average Customer Tenure: {df['customer_tenure_days'].mean():.0f} days ({df['customer_tenure_days'].mean()/365:.1f} years)")
print(f"• New Customers (< 90 days): {len(df[df['customer_tenure_days'] < 90])} ({len(df[df['customer_tenure_days'] < 90])/len(df)*100:.1f}%)")
print(f"• Loyal Customers (> 365 days): {len(df[df['customer_tenure_days'] > 365])} ({len(df[df['customer_tenure_days'] > 365])/len(df)*100:.1f}%)")
print(f"• Average Satisfaction Score: {df['satisfaction_score'].mean():.2f}/10")
print(f"• High Satisfaction (≥ 8): {len(df[df['satisfaction_score'] >= 8])} customers ({len(df[df['satisfaction_score'] >= 8])/len(df)*100:.1f}%)")

# 7. Correlation Insights
print("\n7. CORRELATION INSIGHTS")
print("-" * 80)
purchase_corr = df[numeric_cols_all].corr()['purchase_amount'].sort_values(ascending=False)
print("• Top Factors Influencing Purchase Amount:")
for col, corr in purchase_corr.items():
    if col != 'purchase_amount' and abs(corr) > 0.1:
        direction = "↑" if corr > 0 else "↓"
        print(f"  {direction} {col}: {corr:.4f}")

# 8. Outliers
print("\n8. OUTLIER SUMMARY")
print("-" * 80)
total_outliers = sum([info['count'] for info in outliers_iqr.values()])
print(f"• Total Outlier Instances (IQR method): {total_outliers}")
high_value_sales = df[df['purchase_amount'] > df['purchase_amount'].quantile(0.9)]
print(f"• High-Value Sales (Top 10%): {len(high_value_sales)} transactions totaling ${high_value_sales['purchase_amount'].sum():.2f}")

print("\n" + "="*80)

## 10. Key Findings and Insights Report

In [ ]:
# Visualize outliers using scatter plots with outlier highlighting
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

def highlight_outliers(ax, x_col, y_col, title):
    """Highlight outliers in scatter plot"""
    # Calculate IQR for y column
    Q1 = df[y_col].quantile(0.25)
    Q3 = df[y_col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Identify outliers
    is_outlier = (df[y_col] < lower_bound) | (df[y_col] > upper_bound)
    
    # Plot
    ax.scatter(df[~is_outlier][x_col], df[~is_outlier][y_col], alpha=0.6, 
               color='steelblue', label='Normal', s=50)
    ax.scatter(df[is_outlier][x_col], df[is_outlier][y_col], alpha=0.8, 
               color='red', marker='x', label='Outlier', s=100, linewidths=2)
    
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

highlight_outliers(axes[0, 0], 'age', 'purchase_amount', 'Purchase Amount vs Age (with outliers)')
highlight_outliers(axes[0, 1], 'customer_tenure_days', 'purchase_amount', 'Purchase Amount vs Tenure')
highlight_outliers(axes[1, 0], 'quantity', 'purchase_amount', 'Purchase Amount vs Quantity')
highlight_outliers(axes[1, 1], 'satisfaction_score', 'purchase_amount', 'Purchase Amount vs Satisfaction')

plt.tight_layout()
plt.savefig('../results/08_outlier_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Outlier visualization saved")

In [ ]:
# Detect outliers using IQR method
print("OUTLIER DETECTION (IQR Method)")
print("="*80)
outliers_iqr = explorer.detect_outliers(method='iqr')
for col, info in outliers_iqr.items():
    print(f"\n{col}:")
    print(f"  Outlier Count: {info['count']}")
    print(f"  Percentage: {info['percentage']:.2f}%")

# Detect outliers using Z-score method
print("\n" + "="*80)
print("OUTLIER DETECTION (Z-score Method)")
print("="*80)
outliers_zscore = explorer.detect_outliers(method='zscore')
for col, info in outliers_zscore.items():
    print(f"\n{col}:")
    print(f"  Outlier Count: {info['count']}")
    print(f"  Percentage: {info['percentage']:.2f}%")

## 9. Outlier Detection and Analysis

In [ ]:
# Age distribution analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Create age groups
df['age_group'] = pd.cut(df['age'], bins=[20, 30, 40, 50, 60], 
                          labels=['21-30', '31-40', '41-50', '51-60'])

# Purchase amount by age group
age_group_stats = df.groupby('age_group')['purchase_amount'].agg(['mean', 'median', 'count'])
age_group_stats['mean'].plot(kind='bar', ax=axes[0], color='steelblue', alpha=0.7)
axes[0].set_title('Average Purchase Amount by Age Group', fontweight='bold')
axes[0].set_ylabel('Average Amount ($)')
axes[0].set_xlabel('Age Group')
axes[0].grid(axis='y', alpha=0.3)

# Customer count by age group
age_group_stats['count'].plot(kind='bar', ax=axes[1], color='coral', alpha=0.7)
axes[1].set_title('Customer Count by Age Group', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Age Group')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/07_age_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Age group analysis saved")

In [ ]:
# Violin plots for distribution by category and gender
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Distribution by Product Category
sns.violinplot(data=df, x='product_category', y='purchase_amount', ax=axes[0], palette='Set2')
axes[0].set_title('Purchase Amount Distribution by Product Category', fontweight='bold')
axes[0].set_xlabel('Product Category')
axes[0].set_ylabel('Purchase Amount ($)')
axes[0].grid(axis='y', alpha=0.3)

# Distribution by Gender
sns.violinplot(data=df, x='gender', y='purchase_amount', ax=axes[1], palette='Set1')
axes[1].set_title('Purchase Amount Distribution by Gender', fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Purchase Amount ($)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/06_violin_plots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Violin plots saved")

## 8. Distribution Analysis by Groups

In [ ]:
# Create correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, ax=ax, cbar_kws={'label': 'Correlation Coefficient'},
            linewidths=0.5)
ax.set_title('Correlation Matrix Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/05_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Correlation heatmap saved")

In [ ]:
# Calculate correlation matrix
numeric_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()
correlation_matrix = df[numeric_cols_all].corr()

print("CORRELATION MATRIX")
print("="*80)
print(correlation_matrix)

# Identify high correlations
print("\n" + "="*80)
print("HIGH CORRELATIONS (threshold > 0.7)")
print("="*80)
high_corr = explorer.get_high_correlations(threshold=0.7)
if high_corr:
    for pair, corr_value in high_corr.items():
        print(f"{pair:40} : {corr_value:.4f}")
else:
    print("No correlations above threshold found")

# Identify moderate correlations with purchase_amount
print("\n" + "="*80)
print("CORRELATIONS WITH PURCHASE AMOUNT (sorted)")
print("="*80)
purchase_corr = correlation_matrix['purchase_amount'].sort_values(ascending=False)
print(purchase_corr)

## 7. Correlation Analysis - Key Influencing Factors

In [ ]:
# Bar plots for categorical analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Purchase by Product Category
category_stats = df.groupby('product_category')['purchase_amount'].agg(['sum', 'mean', 'count'])
category_stats['sum'].plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Total Purchase Amount by Category', fontweight='bold')
axes[0, 0].set_ylabel('Total Amount ($)')
axes[0, 0].set_xlabel('Product Category')
axes[0, 0].grid(axis='y', alpha=0.3)

# Average Purchase by Payment Method
payment_stats = df.groupby('payment_method')['purchase_amount'].mean()
payment_stats.plot(kind='bar', ax=axes[0, 1], color='green')
axes[0, 1].set_title('Average Purchase Amount by Payment Method', fontweight='bold')
axes[0, 1].set_ylabel('Average Amount ($)')
axes[0, 1].set_xlabel('Payment Method')
axes[0, 1].grid(axis='y', alpha=0.3)

# Customer Count by Gender
gender_counts = df['gender'].value_counts()
gender_counts.plot(kind='bar', ax=axes[1, 0], color='orange')
axes[1, 0].set_title('Customer Count by Gender', fontweight='bold')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_xlabel('Gender')
axes[1, 0].grid(axis='y', alpha=0.3)

# Average Satisfaction by Category
category_satisfaction = df.groupby('product_category')['satisfaction_score'].mean()
category_satisfaction.plot(kind='bar', ax=axes[1, 1], color='red')
axes[1, 1].set_title('Average Satisfaction Score by Category', fontweight='bold')
axes[1, 1].set_ylabel('Satisfaction Score')
axes[1, 1].set_xlabel('Product Category')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/04_bar_charts.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Bar charts saved")

In [ ]:
# Scatter plots for key relationships
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Purchase Amount vs Customer Tenure
axes[0, 0].scatter(df['customer_tenure_days'], df['purchase_amount'], alpha=0.6, color='steelblue')
axes[0, 0].set_xlabel('Customer Tenure (Days)')
axes[0, 0].set_ylabel('Purchase Amount ($)')
axes[0, 0].set_title('Purchase Amount vs Customer Tenure', fontweight='bold')
axes[0, 0].grid(alpha=0.3)

# Age vs Purchase Amount
axes[0, 1].scatter(df['age'], df['purchase_amount'], alpha=0.6, color='green')
axes[0, 1].set_xlabel('Age')
axes[0, 1].set_ylabel('Purchase Amount ($)')
axes[0, 1].set_title('Purchase Amount vs Age', fontweight='bold')
axes[0, 1].grid(alpha=0.3)

# Satisfaction Score vs Purchase Amount
axes[1, 0].scatter(df['satisfaction_score'], df['purchase_amount'], alpha=0.6, color='orange')
axes[1, 0].set_xlabel('Satisfaction Score')
axes[1, 0].set_ylabel('Purchase Amount ($)')
axes[1, 0].set_title('Purchase Amount vs Satisfaction Score', fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Quantity vs Purchase Amount
axes[1, 1].scatter(df['quantity'], df['purchase_amount'], alpha=0.6, color='red')
axes[1, 1].set_xlabel('Quantity')
axes[1, 1].set_ylabel('Purchase Amount ($)')
axes[1, 1].set_title('Purchase Amount vs Quantity', fontweight='bold')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/03_scatter_plots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Scatter plots saved")

## 6. Bivariate Analysis - Relationships Between Variables

In [ ]:
# Create box plots for outlier detection
fig, axes = plt.subplots(1, 5, figsize=(16, 5))

for idx, col in enumerate(numeric_cols):
    axes[idx].boxplot(df[col], vert=True)
    axes[idx].set_title(f'Boxplot of {col}', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel(col)
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/02_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Boxplot visualization saved")

In [ ]:
# Select numeric columns for distribution analysis
numeric_cols = ['age', 'purchase_amount', 'quantity', 'customer_tenure_days', 'satisfaction_score']

# Create histograms with KDE
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    axes[idx].hist(df[col], bins=25, color='steelblue', alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(axis='y', alpha=0.3)
    
    # Add statistics
    mean_val = df[col].mean()
    median_val = df[col].median()
    axes[idx].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
    axes[idx].axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median: {median_val:.2f}')
    axes[idx].legend()

# Remove extra subplot
fig.delaxes(axes[-1])

plt.tight_layout()
plt.savefig('../results/01_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Distribution plots saved")

## 5. Univariate Analysis - Distribution of Variables

In [ ]:
# Initialize DataExplorer
explorer = DataExplorer(df)

# Get basic statistics
print("BASIC STATISTICS")
print("="*80)
basic_stats = explorer.get_basic_stats()
for key, value in basic_stats.items():
    print(f"{key:20}: {value}")

# Numerical summary
print("\n" + "="*80)
print("NUMERICAL FEATURES SUMMARY")
print("="*80)
numerical_summary = explorer.get_numerical_summary()
print(numerical_summary)

# Additional statistics
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("\n" + "="*80)
print("SKEWNESS AND KURTOSIS")
print("="*80)
for col in numeric_cols:
    print(f"\n{col}:")
    print(f"  Skewness: {skew(df[col]):.4f}")
    print(f"  Kurtosis: {kurtosis(df[col]):.4f}")

## 4. Descriptive Statistics

In [ ]:
# Check for missing values
print("MISSING VALUES ANALYSIS")
print("="*80)
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).values
})
print(missing_data)

# Check for duplicates
print("\n" + "="*80)
print("DUPLICATE ROWS ANALYSIS")
print("="*80)
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows: {duplicate_count}")
print(f"Percentage: {(duplicate_count / len(df) * 100):.2f}%")

# Data type conversion
print("\n" + "="*80)
print("DATA TYPE CONVERSIONS")
print("="*80)

# Convert purchase_date to datetime
df['purchase_date'] = pd.to_datetime(df['purchase_date'])
print(f"✓ Converted 'purchase_date' to datetime")

# Display summary
print("\nData cleaning completed!")
print(f"Dataset is ready for analysis: {df.shape[0]} rows × {df.shape[1]} columns")

## 3. Data Cleaning and Preprocessing

In [ ]:
# Load the dataset
df = pd.read_csv('../data/sales_data.csv')

print("Dataset loaded successfully!")
print(f"\nDataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n" + "="*80)
print("FIRST FEW ROWS")
print("="*80)
print(df.head(10))

print("\n" + "="*80)
print("DATA TYPES AND INFO")
print("="*80)
df.info()

print("\n" + "="*80)
print("COLUMN NAMES AND TYPES")
print("="*80)
for col in df.columns:
    print(f"{col:25} : {df[col].dtype}")

## 2. Load and Inspect the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis
import warnings
warnings.filterwarnings('ignore')

# Configure visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Import custom modules
import sys
sys.path.append('../src')
from eda_utils import DataExplorer
from report_generator import ReportGenerator

print("✓ Libraries imported successfully")

## 1. Import Required Libraries

# Exploratory Data Analysis (EDA) Project
## Sales Data Analysis

This notebook provides a comprehensive exploratory data analysis of sales transaction data, uncovering patterns, trends, and key insights through statistical analysis and visualizations.